In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 3와 Qiskit SDK

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK는 양자 프로그램의 OpenQASM 표현과 QuantumCircuit 클래스 간의 변환을 위한 도구를 제공합니다. 이 도구들은 아직 탐색적 개발 단계에 있으며, Qiskit의 OpenQASM 3로 표현되는 동적 Circuit 기능에 대한 지원이 증가함에 따라 계속 발전할 것입니다.

> **Note:** 이 기능은 아직 탐색적 단계에 있습니다. 따라서 구문과 기능이 변경될 가능성이 있습니다.

## OpenQASM 3 프로그램을 Qiskit으로 가져오기
이 기능을 사용하려면 `qiskit_qasm3_import ` 패키지를 설치해야 합니다. 다음 명령어를 사용하여 설치하세요.

In [1]:
import qiskit.qasm3

program = """
    OPENQASM 3.0;
    include "stdgates.inc";

    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;

    let aliased = q[0:1];

    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }

    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];

    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }

    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }

    out = measure q;
"""
circuit = qiskit.qasm3.loads(program)
circuit.draw("mpl")

<Image src="../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg" alt="Output of the previous code cell" />

현재 OpenQASM 3에서 Qiskit으로 가져오기 위한 두 가지 고수준 함수를 사용할 수 있습니다. 파일 이름을 받는 `load()`와 프로그램 자체를 문자열로 받는 `loads()`입니다:

In [2]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc)

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

이 예시에서는 OpenQASM 3를 사용하여 양자 프로그램을 정의하고, `loads()`를 사용하여 이를 QuantumCircuit으로 직접 변환합니다:

In [3]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f)
f.close()

![이전 코드 셀의 출력](../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg)

## OpenQASM 3로 내보내기
`dumps()`를 사용하여 문자열로 내보내거나, `dump()`를 사용하여 파일로 내보내는 방법으로 Qiskit 코드를 OpenQASM 3로 내보낼 수 있습니다.

### `dumps()` 사용 예시